# lint_tools

> MCP tools for linting notebooks for nbdev best practices

In [ ]:
#| default_exp tools.lint

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from typing import Any, Dict, List, Optional
from pathlib import Path

import re

from rich.panel import Panel
from mcp.server.fastmcp import FastMCP
from mcp.types import ToolAnnotations

from nbdev_mcp.utils.re import (
    RELATIVE_IMPORT_PATTERN,
    DEFAULT_EXP_PATTERN,
    ALL_DEFINITION_PATTERN,
    EXPORT_DIRECTIVE_PATTERN,
    FUNCTION_DEF_PATTERN,
    CLASS_DEF_PATTERN,
    MAIN_GUARD_PATTERN,
    EXPORT_MAIN_PATTERN,
)
from nbdev_mcp.utils.paths import (
    nbs_dir, settings_dict, resolve_selector, with_project, iter_notebooks,
    read_nb, write_nb, abs_module_for_nb, resolve_relative,
)
from nbdev_mcp.utils.nb import join_source, find_default_exp
from nbdev_mcp.utils.rich import render_table, render_panel
from nbdev_mcp.utils.nbformat import (
    read_notebook, write_notebook, has_standard_ending,
    ensure_standard_ending, fix_header_formatting, standardize_notebook,
)

## Validate __init__ Notebooks

In [ ]:
#| export
def validate_inits(
    project: Optional[str] = None,
    fix: bool = False
) -> Dict[str, Any]:
    """Validate that every 00__init__ notebook has correct default_exp.
    
    Accepts numbered-directory prefixes and leaf-only module paths for
    organizational notebook layouts.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    fix : bool
        If True, automatically fix incorrect default_exp values.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'problems' list and 'fixed' count.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    nbs = nbs_dir(p)
    lib = settings_dict(p).get("lib_name") or "pkg"
    lib_path = (p / lib.replace("-", "_")).resolve()
    problems: List[Dict[str, Any]] = []
    fixed = 0
    
    for nb in iter_notebooks(p):
        if nb.name != '00__init__.ipynb':
            continue
        
        rel = nb.relative_to(nbs) if nbs in nb.parents else nb
        parent_parts = list(rel.parent.parts) if len(rel.parts) > 1 else []
        expected_full = '__init__' if len(rel.parts) == 1 else '.'.join(parent_parts + ['__init__'])

        stripped_parts: List[str] = []
        has_numbered = False
        for part in parent_parts:
            match = re.match(r'^\d+_(.+)$', part)
            if match:
                stripped_parts.append(match.group(1))
                has_numbered = True
            else:
                stripped_parts.append(part)
        expected_stripped = '__init__' if len(rel.parts) == 1 else '.'.join(stripped_parts + ['__init__'])
        expected_leaf = '__init__' if len(rel.parts) == 1 else f"{stripped_parts[-1]}.__init__"

        expected_candidates: List[str] = []
        for candidate in (expected_full, expected_stripped, expected_leaf):
            if candidate not in expected_candidates:
                expected_candidates.append(candidate)
        expected_preferred = expected_stripped if has_numbered else expected_full
        
        data = read_nb(nb)
        found = find_default_exp(data)
        found_is_module = False
        if found:
            parts = found.split(".")
            if parts[-1] == "__init__":
                module_path = lib_path.joinpath(*parts[:-1], "__init__.py")
            else:
                module_path = lib_path.joinpath(*parts).with_suffix(".py")
            if module_path.exists():
                found_is_module = True
        
        cell_idx, line_no = -1, -1
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            for j, ln in enumerate(src.splitlines(), 1):
                if re.search(r'#\|\s*default_exp\s+', ln):
                    cell_idx, line_no = i, j
                    break
            if cell_idx != -1:
                break
        
        if found not in expected_candidates and not found_is_module:
            problems.append({
                'notebook': str(nb.relative_to(p)),
                'found': found,
                'expected': expected_preferred,
                'cell': cell_idx,
                'line': line_no
            })
            
            if fix:
                cells = data.get('cells', [])
                if cell_idx != -1:
                    src = join_source(cells[cell_idx].get('source', []))
                    new_src = re.sub(r'(#\|\s*default_exp\s+)[\w\.]+', f'\\1{expected_preferred}', src)
                    cells[cell_idx]['source'] = new_src.splitlines(True)
                else:
                    new_cell = {
                        'cell_type': 'code',
                        'metadata': {},
                        'source': [f'#| default_exp {expected_preferred}\n'],
                        'outputs': [],
                        'execution_count': None
                    }
                    cells.insert(0, new_cell)
                    data['cells'] = cells
                write_nb(nb, data)
                fixed += 1
    
    if problems:
        rows = [[pr['notebook'], str(pr['found']), pr['expected'], str(pr['cell']), str(pr['line'])]
                for pr in problems[:100]]
        pretty = render_table('__init__ default_exp issues', 
                              ['notebook', 'found', 'expected', 'cell', 'line'], rows)
    else:
        pretty = render_panel('validate_inits', 'All 00__init__ notebooks have correct default_exp')
    
    return {'ok': len(problems) == 0, 'problems': problems, 'fixed': fixed, 'pretty': pretty}


## Lint Rules

In [ ]:
#| export
def lint_rules(
    project: Optional[str] = None,
    fix_relative: bool = False
) -> Dict[str, Any]:
    """Lint notebooks for nbdev conventions.
    
    Checks for:
    - Manual __all__ definitions (not allowed)
    - Relative imports (should be absolute)
    - README.md being generated
    - Duplicate exports across notebooks
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    fix_relative : bool
        If True, convert relative imports to absolute.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list and 'modified' count.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    issues: List[Dict[str, Any]] = []
    changed = 0
    
    # Track exports for duplicate detection
    export_locations: Dict[str, List[Dict[str, Any]]] = {}
    
    readme = p / 'README.md'
    if readme.exists():
        issues.append({
            'rule': 'readme_generated',
            'file': str(readme.relative_to(p)),
            'message': 'README.md is generated from nbs/index.ipynb — do not edit directly.'
        })
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        modified = False
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            lines = src.splitlines()
            
            # Check for __all__ definitions
            if re.search(r'^\s*__all__\s*=', src, flags=re.MULTILINE):
                row_lines = [j for j, ln in enumerate(lines, 1) if re.match(r'\s*__all__\s*=', ln)]
                issues.append({
                    'rule': 'no_manual___all__',
                    'notebook': str(nb.relative_to(p)),
                    'cell': i,
                    'lines': row_lines,
                    'message': 'Never define __all__; nbdev auto-generates exports.'
                })
            
            # Track exports for duplicate detection
            has_export = bool(re.search(r'#\|\s*export\s*$', src, re.MULTILINE))
            if has_export:
                # Extract function and class names
                for m in re.finditer(r'^\s*def\s+(\w+)\s*\(', src, re.MULTILINE):
                    name = m.group(1)
                    export_locations.setdefault(name, []).append({
                        'notebook': str(nb.relative_to(p)),
                        'cell': i,
                        'module': mod
                    })
                for m in re.finditer(r'^\s*class\s+(\w+)\s*[\(:]', src, re.MULTILINE):
                    name = m.group(1)
                    export_locations.setdefault(name, []).append({
                        'notebook': str(nb.relative_to(p)),
                        'cell': i,
                        'module': mod
                    })
            
            # Check for relative imports
            new_lines: List[str] = []
            line_changed = False
            for ln in lines:
                m = RELATIVE_IMPORT_PATTERN.match(ln)
                if m:
                    target = m.group(1)
                    abs_mod = resolve_relative(mod, target)
                    abs_imp = f'from {lib}.{abs_mod} import {m.group(2)}'
                    issues.append({
                        'rule': 'no_relative_imports',
                        'notebook': str(nb.relative_to(p)),
                        'cell': i,
                        'line': len(new_lines) + 1,
                        'message': f"Relative import '{ln.strip()}' → '{abs_imp}'",
                        'suggestion': abs_imp
                    })
                    if fix_relative:
                        ln = abs_imp
                        line_changed = True
                new_lines.append(ln)
            
            if fix_relative and line_changed:
                cell['source'] = '\n'.join(new_lines).splitlines(True)
                modified = True
        
        if modified:
            write_nb(nb, data)
            changed += 1
    
    # Add duplicate export issues
    for name, locs in export_locations.items():
        if len(locs) > 1:
            issues.append({
                'rule': 'duplicate_export',
                'symbol': name,
                'locations': locs,
                'message': f"Symbol '{name}' exported in multiple notebooks: {', '.join(loc['notebook'] for loc in locs)}"
            })
    
    rows = []
    for it in issues[:200]:
        loc = it.get('file') or it.get('symbol') or f"{it.get('notebook')}#cell{it.get('cell')}"
        rows.append([it.get('rule', ''), str(loc), it.get('message', '')])
    pretty = render_table('lint issues', ['rule', 'location', 'msg'], rows)
    
    if fix_relative:
        pretty += '\n' + render_panel('relative import fixes', f'Modified notebooks: {changed}')
    
    return {'ok': True, 'issues': issues, 'modified': changed, 'pretty': pretty}


In [ ]:
#| export
def lint_main_guards(project: Optional[str] = None) -> Dict[str, Any]:
    """Detect __main__ guards that would run during nbdev_prepare.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of unsafe main guards.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            lines = src.splitlines()
            
            has_guard = any(
                re.search(r'if\s+__name__\s*==\s*["\']__main__["\']', ln)
                for ln in lines
            )
            if not has_guard:
                continue
            
            has_eval_false = any('#| eval: false' in ln.lower() for ln in lines)
            has_export_main = any(
                re.match(r'\s*#\|\s*export\s+__main__', ln, flags=re.IGNORECASE)
                for ln in lines
            )
            
            if has_eval_false or has_export_main:
                continue
            
            issues.append({
                'notebook': str(nb.relative_to(p)),
                'cell': i,
                'has_eval_false': has_eval_false,
                'has_export_main': has_export_main,
                'message': "Add '#| eval: false' to prevent nbdev_prepare from running main()."
            })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), str(it['has_eval_false']), str(it['has_export_main'])]
                for it in issues[:200]]
        pretty = render_table('__main__ guard warnings', 
                              ['notebook', 'cell', 'eval_false', 'export_main'], rows)
    else:
        pretty = render_panel('lint_main_guards', 'No unsafe __main__ guards detected')
    
    return {'ok': len(issues) == 0, 'issues': issues, 'pretty': pretty}

In [ ]:
#| export
import ast

def lint_imports(
    project: Optional[str] = None,
    ignore_patterns: Optional[List[str]] = None
) -> Dict[str, Any]:
    """Check for unused imports in notebook cells.
    
    Analyzes each code cell to find imports that are never used within that cell.
    Uses AST parsing for accurate detection.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    ignore_patterns : List[str], optional
        List of import names to ignore (e.g., ['typing', 'annotations']).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of unused imports per cell.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    ignore = set(ignore_patterns or [])
    # Always ignore common typing imports and __future__
    ignore.update(['annotations', 'TYPE_CHECKING'])
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Skip cells that are just directives
            if not src.strip() or src.strip().startswith('#|'):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            # Collect all imports
            imported: Dict[str, str] = {}  # alias -> full module name
            for node in ast.walk(tree):
                if isinstance(node, ast.Import):
                    for alias in node.names:
                        name = alias.asname or alias.name.split('.')[0]
                        if name not in ignore:
                            imported[name] = alias.name
                elif isinstance(node, ast.ImportFrom):
                    module = node.module or ''
                    for alias in node.names:
                        name = alias.asname or alias.name
                        if name not in ignore and name != '*':
                            imported[name] = f'{module}.{alias.name}' if module else alias.name
            
            if not imported:
                continue
            
            # Collect all Name usages (excluding imports themselves)
            used_names: set = set()
            for node in ast.walk(tree):
                if isinstance(node, ast.Name):
                    used_names.add(node.id)
                elif isinstance(node, ast.Attribute):
                    # Handle chained attributes like os.path.join
                    n = node
                    while isinstance(n, ast.Attribute):
                        n = n.value
                    if isinstance(n, ast.Name):
                        used_names.add(n.id)
            
            # Find unused imports
            unused = [name for name in imported if name not in used_names]
            
            if unused:
                issues.append({
                    'notebook': str(nb.relative_to(p)),
                    'cell': i,
                    'unused': unused,
                    'message': f"Unused imports: {', '.join(unused)}"
                })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), ', '.join(it['unused'][:5])]
                for it in issues[:200]]
        pretty = render_table('unused imports', ['notebook', 'cell', 'unused'], rows)
    else:
        pretty = render_panel('lint_imports', 'No unused imports found.')
    
    return {
        'ok': len(issues) == 0,
        'issues': issues,
        'total': len(issues),
        'pretty': pretty
    }


In [ ]:
#| export
def lint_types(
    project: Optional[str] = None,
    strict: bool = False
) -> Dict[str, Any]:
    """Check type annotations in notebook cells.
    
    Validates that exported functions and classes have type hints.
    Uses AST parsing to detect missing annotations.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    strict : bool
        If True, require all parameters to have type hints.
        If False (default), only check return type annotations.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of missing type annotations.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Only check exported cells
            if not re.search(r'#\|\s*export\s*$', src, re.MULTILINE):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            for node in ast.walk(tree):
                if isinstance(node, ast.FunctionDef) or isinstance(node, ast.AsyncFunctionDef):
                    func_issues = []
                    
                    # Check return type annotation
                    if node.returns is None:
                        func_issues.append('missing return type')
                    
                    # In strict mode, check parameter annotations
                    if strict:
                        for arg in node.args.args:
                            if arg.arg != 'self' and arg.annotation is None:
                                func_issues.append(f"parameter '{arg.arg}' missing type")
                    
                    if func_issues:
                        issues.append({
                            'notebook': str(nb.relative_to(p)),
                            'cell': i,
                            'function': node.name,
                            'issues': func_issues,
                            'message': f"Function '{node.name}': {', '.join(func_issues)}"
                        })
                
                elif isinstance(node, ast.ClassDef):
                    # Check __init__ method for type hints in strict mode
                    if strict:
                        for item in node.body:
                            if isinstance(item, ast.FunctionDef) and item.name == '__init__':
                                for arg in item.args.args:
                                    if arg.arg not in ('self', 'cls') and arg.annotation is None:
                                        issues.append({
                                            'notebook': str(nb.relative_to(p)),
                                            'cell': i,
                                            'class': node.name,
                                            'function': '__init__',
                                            'issues': [f"parameter '{arg.arg}' missing type"],
                                            'message': f"Class '{node.name}.__init__': parameter '{arg.arg}' missing type"
                                        })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), it.get('function') or it.get('class', ''), it['message'][:60]]
                for it in issues[:200]]
        pretty = render_table('type annotation issues', ['notebook', 'cell', 'name', 'issue'], rows)
    else:
        pretty = render_panel('lint_types', 'All exported functions have type annotations.')
    
    return {
        'ok': len(issues) == 0,
        'issues': issues,
        'total': len(issues),
        'pretty': pretty
    }

In [ ]:
#| export
def auto_fix_lint(
    project: Optional[str] = None,
    fix_relative: bool = True,
    fix_inits: bool = True,
    dry_run: bool = False
) -> Dict[str, Any]:
    """Auto-fix common lint issues in notebooks.
    
    Automatically fixes:
    - Relative imports -> absolute imports
    - Incorrect default_exp in 00__init__.ipynb notebooks
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    fix_relative : bool
        If True, convert relative imports to absolute.
    fix_inits : bool
        If True, fix incorrect default_exp in __init__ notebooks.
    dry_run : bool
        If True, report what would be fixed without making changes.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'fixes' list of applied/proposed fixes.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    fixes: List[Dict[str, Any]] = []
    modified_files = 0
    
    if fix_inits:
        init_result = validate_inits(project=str(p), fix=not dry_run)
        if init_result.get('problems'):
            for prob in init_result['problems']:
                fixes.append({
                    'type': 'init_default_exp',
                    'notebook': prob['notebook'],
                    'found': prob['found'],
                    'expected': prob['expected'],
                    'status': 'would fix' if dry_run else 'fixed'
                })
            if not dry_run:
                modified_files += init_result.get('fixed', 0)
    
    if fix_relative:
        lint_result = lint_rules(project=str(p), fix_relative=not dry_run)
        for issue in lint_result.get('issues', []):
            if issue.get('rule') == 'no_relative_imports':
                fixes.append({
                    'type': 'relative_import',
                    'notebook': issue.get('notebook'),
                    'cell': issue.get('cell'),
                    'message': issue.get('message'),
                    'suggestion': issue.get('suggestion'),
                    'status': 'would fix' if dry_run else 'fixed'
                })
        if not dry_run:
            modified_files += lint_result.get('modified', 0)
    
    if fixes:
        rows = [[f['type'], f.get('notebook', ''), f['status'], f.get('message', '')[:50]]
                for f in fixes[:200]]
        pretty = render_table(f"auto_fix_lint ({'dry run' if dry_run else 'applied'})",
                              ['type', 'notebook', 'status', 'detail'], rows)
    else:
        pretty = render_panel('auto_fix_lint', 'No issues to fix.')
    
    return {
        'ok': True,
        'fixes': fixes,
        'total_fixes': len(fixes),
        'modified_files': modified_files,
        'dry_run': dry_run,
        'pretty': pretty
    }

## `lint_notebook_structure`

In [ ]:
#| export
def lint_notebook_structure(
    project: Optional[str] = None,
    fix: bool = False
) -> Dict[str, Any]:
    """Check and fix notebook structure for nbdev conventions.
    
    Checks that notebooks have:
    - Standard ending structure (## Next, empty export, ## Export, nbdev_export)
    - Properly formatted headers with backticks for code names
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    fix : bool
        If True, automatically fix structure issues.
    
    Returns
    -------
    Dict[str, Any]
        Result dict with:
        - ok: bool - Success status
        - issues: List[Dict] - Structure issues found
        - fixed: int - Number of notebooks fixed (if fix=True)
        - pretty: str - Formatted output
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    fixed_count = 0
    
    for nb_path in iter_notebooks(p):
        nb_issues: List[str] = []
        rel_path = str(nb_path.relative_to(p))
        
        try:
            nb = read_notebook(nb_path)
            
            # Check for standard ending
            if not has_standard_ending(nb):
                nb_issues.append('missing_standard_ending')
            
            # Check for header formatting issues
            code_name_pattern = re.compile(r'^(#+)\s+([a-z_][a-z0-9_]*(?:\([^)]*\))?)(\s*$|\s+\w)', re.IGNORECASE)
            for i, cell in enumerate(nb.cells):
                if cell.cell_type == 'markdown':
                    source = cell.source if isinstance(cell.source, str) else ''.join(cell.source)
                    for line in source.split('\n'):
                        if line.startswith('#'):
                            m = code_name_pattern.match(line)
                            if m and '`' not in line:
                                name = m.group(2)
                                # Check if it looks like a function/class name
                                if '_' in name or name.endswith(')') or name[0].islower():
                                    nb_issues.append(f'unformatted_header_cell_{i}')
                                    break
            
            if nb_issues:
                issues.append({
                    'notebook': rel_path,
                    'issues': nb_issues
                })
                
                if fix:
                    result = standardize_notebook(nb_path, write=True)
                    if result.get('modified'):
                        fixed_count += 1
        
        except Exception as e:
            issues.append({
                'notebook': rel_path,
                'issues': ['error'],
                'error': str(e)
            })
    
    # Build pretty output
    if issues:
        rows = [[issue['notebook'], ', '.join(issue['issues'][:3])] for issue in issues[:50]]
        status = f"{len(issues)} notebooks with issues"
        if fix:
            status += f", {fixed_count} fixed"
        pretty = render_table(f"lint_notebook_structure: {status}", 
                              ['notebook', 'issues'], rows)
    else:
        pretty = render_panel('lint_notebook_structure', 'All notebooks have correct structure.')
    
    return {
        'ok': True,
        'issues': issues,
        'total_issues': len(issues),
        'fixed': fixed_count,
        'pretty': pretty
    }

In [ ]:
#| export
def lint_private_attributes(
    project: Optional[str] = None,
    allow_patterns: Optional[List[str]] = None
) -> Dict[str, Any]:
    """Flag private attributes (self._xxx) in exported classes.
    
    MCP tools should be stateless. Private attributes suggest hidden state
    that persists between calls, which breaks the MCP model.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    allow_patterns : List[str], optional
        List of attribute name patterns to allow (e.g., ['_cache', '_lock']).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of private attribute usages.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    allowed = set(allow_patterns or [])
    issues: List[Dict[str, Any]] = []
    
    private_attr_pattern = re.compile(r'self\.(_\w+)')
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            if not re.search(r'#\|\s*export\s*$', src, re.MULTILINE):
                continue
            
            for line_no, line in enumerate(src.splitlines(), 1):
                for m in private_attr_pattern.finditer(line):
                    attr_name = m.group(1)
                    if attr_name not in allowed:
                        try:
                            tree = ast.parse(src)
                            in_class = any(isinstance(node, ast.ClassDef) for node in ast.walk(tree))
                            if in_class:
                                issues.append({
                                    'notebook': str(nb.relative_to(p)),
                                    'cell': i,
                                    'line': line_no,
                                    'attribute': attr_name,
                                    'message': f"Private attribute {attr_name} found. MCP tools should be stateless."
                                })
                        except SyntaxError:
                            pass
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), str(it['line']), it['attribute'], it['message'][:40]]
                for it in issues[:100]]
        pretty = render_table(
            'Private Attribute Issues',
            ['Notebook', 'Cell', 'Line', 'Attribute', 'Message'],
            rows
        )
    else:
        pretty = render_panel('lint_private_attributes', 'No private attributes found in exported classes')
    
    return {'ok': True, 'issues': issues, 'count': len(issues), 'pretty': pretty}

In [ ]:
#| export
def lint_cell_complexity(
    project: Optional[str] = None,
    max_classes: int = 1,
    max_functions: int = 2
) -> Dict[str, Any]:
    """Flag cells with too many definitions (classes/functions).
    
    nbdev encourages one class or function per cell for:
    - Granular testing
    - Clear cell-level documentation
    - Literate programming readability
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    max_classes : int
        Maximum classes allowed per cell (default: 1).
    max_functions : int
        Maximum functions allowed per cell (default: 2).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of overly complex cells.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            if not re.search(r'#\|\s*export\s*$', src, re.MULTILINE):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            classes = [n for n in ast.iter_child_nodes(tree) if isinstance(n, ast.ClassDef)]
            functions = [n for n in ast.iter_child_nodes(tree) if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]
            
            if len(classes) > max_classes or len(functions) > max_functions:
                issues.append({
                    'notebook': str(nb.relative_to(p)),
                    'cell': i,
                    'classes': len(classes),
                    'functions': len(functions),
                    'class_names': [c.name for c in classes],
                    'function_names': [f.name for f in functions],
                    'message': f"Cell has {len(classes)} classes and {len(functions)} functions. Consider splitting."
                })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), str(it['classes']), str(it['functions']), it['message'][:50]]
                for it in issues[:100]]
        pretty = render_table(
            'Cell Complexity Issues',
            ['Notebook', 'Cell', 'Classes', 'Functions', 'Message'],
            rows
        )
    else:
        pretty = render_panel('lint_cell_complexity', 'All cells have appropriate complexity')
    
    return {'ok': True, 'issues': issues, 'count': len(issues), 'pretty': pretty}

In [ ]:
#| export
def lint_import_order(
    project: Optional[str] = None
) -> Dict[str, Any]:
    """Check that `from __future__ import` is in the first export cell.
    
    Python requires __future__ imports to be at the top of a module.
    In nbdev, this means the first export cell must contain any
    __future__ imports, before other export cells.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias.
    
    Returns
    -------
    Dict[str, Any]
        Result with 'issues' list of import order violations.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    issues: List[Dict[str, Any]] = []
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        first_export_cell = None
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            if re.search(r'#\|\s*export\s*$', src, re.MULTILINE):
                if first_export_cell is None:
                    first_export_cell = i
                elif 'from __future__ import' in src:
                    issues.append({
                        'notebook': str(nb.relative_to(p)),
                        'cell': i,
                        'first_export_cell': first_export_cell,
                        'message': f"__future__ import should be in first export cell (cell {first_export_cell}), not cell {i}"
                    })
    
    if issues:
        rows = [[it['notebook'], str(it['cell']), str(it['first_export_cell']), it['message'][:60]]
                for it in issues[:100]]
        pretty = render_table(
            'Import Order Issues',
            ['Notebook', 'Cell', 'First Export', 'Message'],
            rows
        )
    else:
        pretty = render_panel('lint_import_order', 'All __future__ imports are correctly placed')
    
    return {'ok': True, 'issues': issues, 'count': len(issues), 'pretty': pretty}

## MCP Registration

In [ ]:
#| export
# Tool annotation definitions for lint tools
LINT_TOOL_ANNOTATIONS = {
    'validate_inits': ToolAnnotations(
        title="Validate Inits",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_rules': ToolAnnotations(
        title="Lint Rules",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_main_guards': ToolAnnotations(
        title="Lint Main Guards",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_imports': ToolAnnotations(
        title="Lint Imports",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_types': ToolAnnotations(
        title="Lint Types",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'auto_fix_lint': ToolAnnotations(
        title="Auto Fix Lint",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_notebook_structure': ToolAnnotations(
        title="Lint Notebook Structure",
        readOnlyHint=False,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_private_attributes': ToolAnnotations(
        title="Lint Private Attributes",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_cell_complexity': ToolAnnotations(
        title="Lint Cell Complexity",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_import_order': ToolAnnotations(
        title="Lint Import Order",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
    'lint_dead_exports': ToolAnnotations(
        title="Lint Dead Exports",
        readOnlyHint=True,
        idempotentHint=True,
        openWorldHint=False
    ),
}

def add_lint_tools(mcp: FastMCP) -> None:
    """Register all lint tools with the MCP server.
    
    Parameters
    ----------
    mcp : FastMCP
        The MCP server instance to register tools with.
    """
    tools = [
        ('validate_inits', validate_inits),
        ('lint_rules', lint_rules),
        ('lint_main_guards', lint_main_guards),
        ('lint_imports', lint_imports),
        ('lint_types', lint_types),
        ('auto_fix_lint', auto_fix_lint),
        ('lint_notebook_structure', lint_notebook_structure),
        ('lint_private_attributes', lint_private_attributes),
        ('lint_cell_complexity', lint_cell_complexity),
        ('lint_import_order', lint_import_order),
        ('lint_dead_exports', lint_dead_exports),
    ]
    
    for name, func in tools:
        annotations = LINT_TOOL_ANNOTATIONS.get(name)
        mcp.tool(name=name, annotations=annotations)(func)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

In [ ]:
#| export
def lint_dead_exports(
    project: Optional[str] = None,
    ignore_patterns: Optional[List[str]] = None
) -> Dict[str, Any]:
    """Find exported symbols that are never imported elsewhere.
    
    Scans all notebooks to find exports that no other notebook imports.
    These are potential dead code that could be removed.
    
    Parameters
    ----------
    project : str, optional
        Project path or alias. Uses current project if not specified.
    ignore_patterns : List[str], optional
        List of symbol name patterns to ignore (e.g., ['main', 'cli']).
    
    Returns
    -------
    Dict[str, Any]
        Result with 'dead_exports' list of unused symbols.
    """
    try:
        p = resolve_selector(project)
    except Exception as e:
        return {'ok': False, 'error': str(e)}
    
    s = settings_dict(p)
    lib = s.get('lib_name') or 'pkg'
    ignore = set(ignore_patterns or [])
    # Common entry points that may not be imported
    ignore.update(['main', 'cli', 'app', 'run', 'serve'])
    
    # Phase 1: Collect all exports per module
    exports_by_module: Dict[str, Dict[str, Dict[str, Any]]] = {}  # module -> {symbol -> {notebook, cell}}
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        mod = find_default_exp(data) or abs_module_for_nb(p, nb)[1]
        fq_mod = f'{lib}.{mod}'
        
        if fq_mod not in exports_by_module:
            exports_by_module[fq_mod] = {}
        
        for i, cell in enumerate(data.get('cells', [])):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            # Only check exported cells
            if not re.search(r'#\|\s*export\s*$', src, re.MULTILINE):
                continue
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            # Collect top-level definitions
            for node in ast.iter_child_nodes(tree):
                if isinstance(node, ast.FunctionDef) or isinstance(node, ast.AsyncFunctionDef):
                    name = node.name
                    if not name.startswith('_') and name not in ignore:
                        exports_by_module[fq_mod][name] = {
                            'notebook': str(nb.relative_to(p)),
                            'cell': i,
                            'type': 'function'
                        }
                elif isinstance(node, ast.ClassDef):
                    name = node.name
                    if not name.startswith('_') and name not in ignore:
                        exports_by_module[fq_mod][name] = {
                            'notebook': str(nb.relative_to(p)),
                            'cell': i,
                            'type': 'class'
                        }
                elif isinstance(node, ast.Assign):
                    for target in node.targets:
                        if isinstance(target, ast.Name):
                            name = target.id
                            if not name.startswith('_') and name.isupper() and name not in ignore:
                                exports_by_module[fq_mod][name] = {
                                    'notebook': str(nb.relative_to(p)),
                                    'cell': i,
                                    'type': 'constant'
                                }
    
    # Phase 2: Collect all imports across notebooks
    imported_symbols: set = set()  # (module, symbol) pairs
    imported_modules: set = set()  # modules imported with "import module"
    
    for nb in iter_notebooks(p):
        data = read_nb(nb)
        
        for cell in data.get('cells', []):
            if cell.get('cell_type') != 'code':
                continue
            src = join_source(cell.get('source', []))
            
            try:
                tree = ast.parse(src)
            except SyntaxError:
                continue
            
            for node in ast.walk(tree):
                if isinstance(node, ast.ImportFrom):
                    mod = node.module or ''
                    if mod.startswith(lib):
                        for alias in node.names:
                            if alias.name == '*':
                                # import * means all exports are used
                                imported_modules.add(mod)
                            else:
                                imported_symbols.add((mod, alias.name))
                elif isinstance(node, ast.Import):
                    for alias in node.names:
                        if alias.name.startswith(lib):
                            imported_modules.add(alias.name)
    
    # Phase 3: Find dead exports (not imported anywhere)
    dead_exports: List[Dict[str, Any]] = []
    
    for fq_mod, symbols in exports_by_module.items():
        # If module is imported with "import module", all symbols are potentially used
        if fq_mod in imported_modules:
            continue
        
        for sym_name, sym_info in symbols.items():
            # Check if symbol is imported anywhere
            is_used = any(
                (mod, sym_name) in imported_symbols
                for mod in [fq_mod, fq_mod.replace(lib + '.', '')]
            )
            
            if not is_used:
                dead_exports.append({
                    'symbol': sym_name,
                    'module': fq_mod,
                    'notebook': sym_info['notebook'],
                    'cell': sym_info['cell'],
                    'type': sym_info['type'],
                    'message': f"'{sym_name}' exported from {fq_mod} but never imported"
                })
    
    # Sort by notebook, then symbol
    dead_exports.sort(key=lambda x: (x['notebook'], x['symbol']))
    
    if dead_exports:
        rows = [[it['symbol'], it['type'], it['notebook'], it['module']]
                for it in dead_exports[:200]]
        pretty = render_table(
            f'Dead Exports: {len(dead_exports)} unused symbols',
            ['Symbol', 'Type', 'Notebook', 'Module'],
            rows
        )
    else:
        pretty = render_panel('lint_dead_exports', 'No dead exports found - all exports are used.')
    
    return {
        'ok': len(dead_exports) == 0,
        'dead_exports': dead_exports,
        'total': len(dead_exports),
        'total_exports': sum(len(syms) for syms in exports_by_module.values()),
        'pretty': pretty
    }